In [1]:
import pandas as pd
import numpy as np

In [2]:
# Load data
sessions_featured = pd.read_csv('../data_csv/4 sessions_featured.csv')
sessions_modeled = pd.read_csv('../data_csv/5 sessions_modeled_lgb.csv')
events = pd.read_csv('../data_csv/2 events_cleaned.csv', usecols=[
    'user_session', 'product_id', 'category_id', 'event_type', 'price', 'brand'
])

sessions = pd.concat([sessions_featured, sessions_modeled[['purchase_prob', 'segment']]], axis=1)

In [3]:
# ── Target Group 1: High Intent non-converting (Cart → Purchase gap)
target_cart = sessions[
    (sessions['segment'] == 'High Intent') &
    (sessions['converted'] == 0) &
    (sessions['num_cart'] > 0)
]['user_session']

# ── Target Group 2: View-only non-converting (View → Cart gap)
target_view = sessions[
    (sessions['segment'] == 'High Intent') &
    (sessions['converted'] == 0) &
    (sessions['num_cart'] == 0)
]['user_session']

print(f"Target Group 1 (cart but no purchase): {len(target_cart):,}")
print(f"Target Group 2 (view only, no cart):   {len(target_view):,}")

Target Group 1 (cart but no purchase): 309,658
Target Group 2 (view only, no cart):   2,502


In [4]:
# Get events for target cart sessions
target_cart_events = events[events['user_session'].isin(target_cart)]

# Find what they added to cart
cart_items = target_cart_events[target_cart_events['event_type'] == 'cart'][
    ['user_session', 'product_id', 'category_id', 'price', 'brand']
]

print(f"Cart events from target sessions: {len(cart_items):,}")
print(cart_items.head())

Cart events from target sessions: 2,648,811
                            user_session  product_id          category_id  \
6   81326ac6-daa4-4f0a-b488-fd0956a78733     5739055  1487580008246412266   
19  618f3d7d-2939-47ea-8f1d-07a4f97d0fe2     5859414  1487580005671109489   
21  618f3d7d-2939-47ea-8f1d-07a4f97d0fe2     5859413  1487580005671109489   
25  8bbff347-0be1-470e-8860-d9a75db965b2     5677366  1487580008246412266   
26  618f3d7d-2939-47ea-8f1d-07a4f97d0fe2     5859411  1487580005671109489   

    price   brand  
6    4.75  kapous  
19   2.37  masura  
21   2.37  masura  
25   7.30   estel  
26   2.37  masura  


In [5]:
# Find top selling products per category (from all purchase events)
purchase_events = events[events['event_type'] == 'purchase']

top_products_by_cat = purchase_events.groupby(['category_id', 'product_id', 'brand']).agg(
    purchase_count=('event_type', 'count'),
    avg_price=('price', 'mean')
).reset_index().sort_values('purchase_count', ascending=False)

print(f"Total products with purchase history: {len(top_products_by_cat):,}")
print(top_products_by_cat.head(10))

Total products with purchase history: 42,387
               category_id  product_id    brand  purchase_count  avg_price
33515  1602943681873052386     5809910  grattol            7549   5.193068
21042  1487580009445982239     5854897    irisk            4630   0.317981
20331  1487580009286598681     5700037   runail            3682   0.396306
20391  1487580009286598681     5802432  unknown            3533   0.319992
3513   1487580005268456287     5751422      uno            3521  10.857657
33517  1602943681873052386     5809912  grattol            3307   5.178676
9529   1487580006317032337     5815662  unknown            3246   0.911389
21094  1487580009471148064        5304   runail            3133   0.317229
2296   1487580005092295511     5751383      uno            2948  10.223301
2622   1487580005092295511     5849033      uno            2782  10.214515


In [6]:
# Get products already viewed by target sessions
viewed_items = target_cart_events[target_cart_events['event_type'] == 'view'][
    ['user_session', 'product_id']
].drop_duplicates()

print(f"Viewed products across target sessions: {len(viewed_items):,}")

# Get top 3 products per category (excluding already viewed)
top3_per_cat = top_products_by_cat.groupby('category_id').head(3).reset_index(drop=True)

# For each session, find cart categories, then recommend unseen top products
def get_recommendations(session_id, cart_items, viewed_items, top3_per_cat, n=3):
    # Categories in cart
    session_cats = cart_items[cart_items['user_session'] == session_id]['category_id'].unique()
    
    # Products already viewed
    already_viewed = viewed_items[viewed_items['user_session'] == session_id]['product_id'].tolist()
    
    recs = []
    for cat in session_cats:
        candidates = top3_per_cat[
            (top3_per_cat['category_id'] == cat) &
            (~top3_per_cat['product_id'].isin(already_viewed))
        ].head(n)
        
        if len(candidates) > 0:
            candidates = candidates.copy()
            candidates['source_session'] = session_id
            recs.append(candidates)
    
    return pd.concat(recs) if recs else pd.DataFrame()

# Test on sample of 500 sessions
sample_sessions = target_cart.sample(500, random_state=46).tolist()
all_recs = []
for sid in sample_sessions:
    rec = get_recommendations(sid, cart_items, viewed_items, top3_per_cat)
    if len(rec) > 0:
        all_recs.append(rec)

recommendations = pd.concat(all_recs, ignore_index=True)
print(f"\nRecommendations generated: {len(recommendations):,}")
print(f"Sessions covered: {recommendations['source_session'].nunique():,}")
print(recommendations.head(10))

Viewed products across target sessions: 1,797,387

Recommendations generated: 4,906
Sessions covered: 499
           category_id  product_id    brand  purchase_count  avg_price  \
0  1602943681873052386     5809910  grattol            7549   5.193068   
1  1602943681873052386     5809912  grattol            3307   5.178676   
2  1602943681873052386     5809911  grattol            1827   5.205233   
3  1487580005092295511     5814046  grattol            1155   6.247004   
4  1487580012021285001     5820083    domix             198   1.110000   
5  1487580012021285001     5685340  farmona             115  16.830000   
6  1487580005092295511     5751383      uno            2948  10.223301   
7  1487580005092295511     5849033      uno            2782  10.214515   
8  1487580005092295511     5814046  grattol            1155   6.247004   
9  1487580005268456287     5751422      uno            3521  10.857657   

                         source_session  
0  100791fa-0ada-4675-96fb-e944d5d502

In [8]:
# Run on 1000 sample sessions
sample_1000 = target_cart.sample(1000, random_state=46).tolist()
all_recs = []

for sid in sample_1000:
    rec = get_recommendations(sid, cart_items, viewed_items, top3_per_cat)
    if len(rec) > 0:
        all_recs.append(rec)

final_recommendations = pd.concat(all_recs, ignore_index=True)
final_recommendations.to_csv('recommendations_sample.csv', index=False)

print(f"Total recommendations: {len(final_recommendations):,}")
print(f"Sessions covered: {final_recommendations['source_session'].nunique():,}")
print(f"Coverage rate: {final_recommendations['source_session'].nunique()/1000:.1%}")

Total recommendations: 9,683
Sessions covered: 998
Coverage rate: 99.8%
